## 第23章　梯度爆炸与梯度裁剪 —— 参数"飞走"怎么办

> 本章目标：亲手模拟梯度爆炸——循环乘大数、loss 从正常跳到 NaN——然后用一行 `clip_grad_norm_` 把梯度"摁住"。理解为什么 RNN/LSTM 训练中梯度裁剪是必选项，以及 Transformer 中即使有 LayerNorm 也保留裁剪作为最后防线。
> 前置知识：第 12 章（梯度下降）、第 13 章（链式法则）、第 20 章（浮点精度）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")


### 23.1　模拟梯度爆炸 —— 循环乘大数

梯度爆炸不是教科书上的传说——它在真实训练中频繁发生。根本原因只有一个：**链式法则中的因子大于 1，层层相乘后指数级放大。**

想象一个 50 层的网络，每层的本地梯度平均为 1.5。反向传播时梯度 = 1.5⁵⁰ ≈ 6.4×10⁸。在 float32 下，梯度从正常值跳到数百万、再到 Inf、再到 NaN——整个过程可能只需要几个 step。

> **关键洞察**：梯度爆炸和梯度消失是同一枚硬币的两面——都是链式法则中"因子连乘"的必然结果。因子 > 1 → 爆炸，因子 < 1 → 消失。深层网络同时遭受两者的折磨。

---


### 23.2　损失震荡可视化 —— 亲手炸一次

用代码模拟这条爆炸曲线：从一个初始梯度出发，每"层"乘一个略大于 1 的因子，亲眼看到它在几十步内从 1.0 飙升到 10⁹。同时也模拟消失方向——因子 < 1 时梯度迅速归零。

💻 **代码　模拟深层网络中的梯度爆炸曲线**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n_layers = 50
grad_init = 1.0

# 模拟三种场景
scenarios = {
    "稳定 (因子≈1.0)": 1.0 + np.random.randn(n_layers) * 0.05,   # 均值=1, 微有波动
    "爆炸 (因子≈1.5)": 1.5 + np.random.randn(n_layers) * 0.1,    # 均值>1 → 爆炸!
    "消失 (因子≈0.5)": 0.5 + np.random.randn(n_layers) * 0.05,   # 均值<1 → 消失
}

fig, ax = plt.subplots(figsize=(10, 5))
for label, factors in scenarios.items():
    grad = grad_init
    grad_history = [grad]
    for f in factors:
        grad = grad * f
        grad_history.append(grad)
    ax.plot(range(n_layers + 1), grad_history, linewidth=2, label=label)

ax.set_yscale('log')
ax.set_xlabel('网络层数 (反向传播方向)'); ax.set_ylabel('梯度范数 (log scale)')
ax.set_title('梯度爆炸 vs 梯度消失：链式法则的放大/缩小效应')
ax.axhline(y=1e6, color='red', linestyle='--', alpha=0.5, label='float16 溢出边界')
ax.legend(); ax.grid(alpha=0.3); plt.show()

print("爆炸方向：5层后梯度已经 1.5^5 ≈ 7.6, 50层后 ≈ 6.4×10^8")
print("消失方向：50层后梯度 ≈ 0.5^50 ≈ 8.9×10^-16 → 第20章的小梯度陷阱!")


> **关键洞察**：梯度爆炸和梯度消失是同一枚硬币的两面——都是链式法则中"因子连乘"的必然结果。因子 > 1 → 爆炸（前层梯度巨大），因子 < 1 → 消失（前层梯度为零）。深层网络同时遭受两者的折磨：靠近输出的层梯度适中，靠近输入的层要么炸穿、要么归零。

🔗 **AI 连接**：第 21 章的 LayerNorm 和 ResNet/Transformer 的残差连接（第 30 章）都是为了把链式法则因子"压"回 1 附近——让梯度不再指数级放大或缩小。

---


### 23.3　梯度裁剪 —— 一行代码把梯度"摁住"

梯度裁剪（Gradient Clipping）是最简单粗暴但极其有效的解决方案：**在每次参数更新前，如果梯度的 L2 范数超过一个阈值，就按比例缩回去。**

$$\text{if } \|g\| > \text{max\_norm}, \quad g = g \cdot \frac{\text{max\_norm}}{\|g\|}$$

翻译成人话：梯度方向不变（仍然指向最陡下降），但步长被限制了——不管梯度多大，单步更新量不会超过阈值。

📐 **定义　梯度裁剪（Gradient Clipping）**：`torch.nn.utils.clip_grad_norm_(parameters, max_norm)`。对全体参数的梯度向量求 L2 范数，若超过 max_norm 则等比例缩放。典型阈值：RNN 用 0.5~5，Transformer 用 1.0。

💻 **代码　模拟训练中的梯度爆炸 + 裁剪效果对比**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n_steps = 100
true_param = 3.0
param_no_clip = 0.0
param_with_clip = 0.0
lr = 0.01
max_norm = 1.0

history_no_clip = [param_no_clip]
history_with_clip = [param_with_clip]
grad_norms = []

for step in range(n_steps):
    # 模拟梯度：大部分正常，偶尔爆炸
    grad = 2 * (param_no_clip - true_param)  # 正常梯度
    if step % 20 == 0:  # 每 20 步来一次爆炸（模拟异常 batch）
        grad = grad * np.exp(np.random.uniform(5, 10))  # ×150~22000

    grad_norm = abs(grad)
    grad_norms.append(grad_norm)

    # 无裁剪
    param_no_clip -= lr * grad

    # 有裁剪：梯度范数超过 max_norm 就缩放
    if grad_norm > max_norm:
        grad_clipped = grad * (max_norm / grad_norm)
    else:
        grad_clipped = grad
    param_with_clip -= lr * grad_clipped

    history_no_clip.append(param_no_clip)
    history_with_clip.append(param_with_clip)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(n_steps + 1), history_no_clip, 'r-', linewidth=1.5, label='无裁剪')
axes[0].plot(range(n_steps + 1), history_with_clip, 'b-', linewidth=2, label='有裁剪 (max=1.0)')
axes[0].axhline(y=true_param, color='green', linestyle='--', label=f'最优值={true_param}')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('参数值')
axes[0].set_title('参数收敛轨迹：无裁剪 vs 有裁剪')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].bar(range(n_steps), grad_norms, width=1.0, color='steelblue', alpha=0.7)
axes[1].axhline(y=max_norm, color='red', linestyle='--', linewidth=2, label=f'裁剪阈值={max_norm}')
axes[1].set_yscale('log'); axes[1].set_xlabel('Step'); axes[1].set_ylabel('梯度范数 (log)')
axes[1].set_title('梯度范数变化：爆炸 step 远超阈值')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("无裁剪：爆炸 step 把参数'炸飞'到几百甚至 NaN，之前的训练全部作废")
print("有裁剪：梯度方向保持，但步长被限制 → 参数稳步收敛到 3.0")


> **关键洞察**：梯度裁剪的巧妙之处在于——它**只缩放梯度的幅度，不改变其方向**。这意味着即使在爆炸 step，参数更新仍然指向正确的大方向，只是步长被限制在了安全范围内。这是一种"宁可走慢点，绝不飞出去"的保守策略，在 RNN 训练中是不可或缺的。

🔗 **AI 连接**：PyTorch 训练循环的标准模板中，`clip_grad_norm_` 紧挨在 `optimizer.step()` 之前。第 31 章训练循环全景中你会看到完整的顺序：`loss.backward()` → `clip_grad_norm_` → `optimizer.step()` → `optimizer.zero_grad()`。

---


### 23.4　为什么 RNN/LSTM 必须裁剪 —— Transformer 的视角

RNN/LSTM 是按时间步展开的——一个长度为 100 的序列，等价于一个 100 层的"深层网络"，每一层共享相同的权重矩阵 **W**。如果 **W** 的最大特征值 > 1，梯度沿时间反向传播时指数爆炸；如果 < 1，梯度指数消失。

这就是为什么 RNN 时代梯度裁剪是标配——没有裁剪，RNN 几乎无法训练超过 20 步的序列。

**Transformer 为什么相对安全但仍保留裁剪？** Transformer 的注意力机制让任意两个位置可以直接交互（不需要逐时间步传递），大大缩短了梯度传播路径。但 Transformer 仍然非常深（几十到上百层），每层内部的 FFN 和 Attention 都可能产生不稳定的梯度。所以现代 Transformer 训练中，梯度裁剪作为**最后的安全网**通常设为 `max_norm=1.0`。

---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）为什么深层网络中梯度容易爆炸或消失？用链式法则的"因子连乘"来解释。
2. （代码）模拟一个 20 层线性网络的梯度传播：初始化权重矩阵为随机值 × 缩放因子 `scale`。分别测试 `scale=1.5`（爆炸）和 `scale=0.5`（消失），画出每层的梯度范数曲线，标注 float32 的有效范围。
---
> 🔗 **章末钩子**：梯度裁剪是"治标"——在爆炸发生时把梯度摁回去。但更好的方案是从源头防止梯度爆炸：让每一步的更新步长自适应地调整——在平缓的地方迈大步，在陡峭的地方迈小步。这就是优化算法要做的事。
> 预览：下一章——**优化算法：从 SGD 到 AdamW**，训练加速的工程智慧。
